In [22]:
#Seurat Core
library(Seurat)
#library(BPCells)
library(SeuratWrappers)
library(S4Vectors)

#non-Saitja QC libraries
library(DropletUtils)
library(scDblFinder)
library(SoupX)
library(SingleCellExperiment)

#Visualizations
library(ggplot2)
library(patchwork)
library(dplyr)
library(tidyverse)
library(glue)

#utility function
source("utility_scripts.R")

In [32]:
gse176031.obj <- readRDS("./sc_rds_files/gse176031.rds")

In [36]:
object <- gse176031.obj
dataset_name <- "gse176031"

In [37]:
identical(rownames(gse176031.obj@meta.data), 
          colnames(gse176031.obj@assays$RNA))

[1] TRUE

In [38]:
scqc_scatters <- function(tmp_object,
                        dataset_name = "example_dataset"){

    qc_images <- "qc_images"
    if(!dir.exists(qc_images)){
        dir.create("qc_images")
    }
    
    QC_testing <- tmp_object

    #percentage of mitochondrial RNA counts
    #calculate percentageof mitochondrial RNA per barcode
    QC_testing[["percent.mt"]] <- PercentageFeatureSet(QC_testing, pattern = "^MT-")
    #percentage of ribosomal RNA counts
    QC_testing[["percent.ribo"]] <- PercentageFeatureSet(QC_testing, pattern = "^RP[SL]")
    #What is the complexity of each cell library
    QC_testing$log10GenesPerUMI <- log10(QC_testing$nFeature_RNA) / log10(QC_testing$nCount_RNA)

    # Summary statistics to guide threshold setting
    cat("\nQC Metric Distributions:\n")
    cat("nCount_RNA (UMI):\n")
    cat("  Median:", median(QC_testing$nCount_RNA), 
        "| Q1-Q3:", quantile(QC_testing$nCount_RNA, 0.25), "-", 
        quantile(QC_testing$nCount_RNA, 0.75), "\n")
    
    cat("nFeature_RNA (genes):\n")
    cat("  Median:", median(QC_testing$nFeature_RNA),
        "| Q1-Q3:", quantile(QC_testing$nFeature_RNA, 0.25), "-",
        quantile(QC_testing$nFeature_RNA, 0.75), "\n")
    
    cat("percent.mt:\n")
    cat("  Median:", round(median(QC_testing$percent.mt), 2), "%",
        "| 95th percentile:", round(quantile(QC_testing$percent.mt, 0.95), 2), "%\n")
    
    cat("percent.ribo:\n")
    cat("  Median:", round(median(QC_testing$percent.ribo), 2), "%\n")

    #setting Idents to the original identity of the object
    Idents(QC_testing) <- "orig.ident"

    p4 <- VlnPlot(
      QC_testing,
      features =c("nCount_RNA", "nFeature_RNA", "percent.mt", "percent.ribo"),
      ncol = 4,
      pt.size = 0.1
    ) + theme(plot.title = element_text(face = "bold"))

    
    #file number 1 for violin plots
    file_1 <-  glue::glue("./qc_images/{dataset_name}_UMI_Feature_MT_scatters_pre-empty.png")
    gg_patchwork(p4, file_1, width = 10, height = 6)
    
    
    #FeatureScatter, UMI vs Gene Detected
    #
    p5 <- FeatureScatter(QC_testing, feature1 = "nCount_RNA", feature2 = "nFeature_RNA") +
      labs(title = "UMI vs Genes Detected")

    #FeatureScatter, UMI vs Mitochondrial %
    p6 <- FeatureScatter(QC_testing, feature1 = "nCount_RNA", feature2 = "percent.mt") +
      labs(title = "UMI vs Mitochondrial %")

    # Mitochondrial % vs Ribosomal %
    #How many dying cells vs active cells
    p7 <- FeatureScatter(QC_testing, feature1 = "percent.mt", feature2 = "percent.ribo") +
      labs(title = "Mitochondrial % vs Ribosomal %")
    
    p_scatter <- p5 + p6 + p7
    
    file_2 <- glue::glue("./qc_images/{dataset_name}_relation_scatters.png")
    gg_patchwork(p_scatter, file_2, width = 12, height = 6)

    cat("Visual Credits go Dr. Li Guo of www.ngs101.com")
    
    return(list(vlnplots = p4, rltn_scatters = p_scatter))
}#end of function

In [40]:
scqc_scatters(object, dataset_name)


QC Metric Distributions:
nCount_RNA (UMI):
  Median: 1138 | Q1-Q3: 237 - 2510 
nFeature_RNA (genes):
  Median: 716 | Q1-Q3: 125 - 1392 
percent.mt:


ERROR: Error in quantile.default(QC_testing$percent.mt, 0.95): missing values and NaN's not allowed if 'na.rm' is FALSE


In [17]:
scqc_thresholds <- function(tmp_object,
                            dataset_name = "example",
                            nfeature_min = 200,
                            nfeature_max = 7500,
                            ncount_min = 500,
                            mt_thresh = 10){
    
    qc_df <- data.frame(
                nCount_RNA = QC_testing$nCount_RNA,
                nFeature_RNA = QC_testing$nFeature_RNA,
                percent.mt = QC_testing$percent.mt
                )#setting dataframe for qc filters
    
    qc_df$pass_qc <- (
              qc_df$nCount_RNA >= ncount_min &
                qc_df$nCount_RNA <= ncount_max &
                qc_df$nFeature_RNA >= nfeature_min &
                qc_df$nFeature_RNA <= nfeature_max &
                qc_df$percent.mt < mt_thresh
                )# setting boundary filters for qc data

    #Threshol
    p8 <- ggplot(qc_df, aes(x = log10(nCount_RNA + 1), y = log10(nFeature_RNA + 1), 
                        color = pass_qc)) +
                  geom_point(alpha = 0.5, size = 1) +
                  geom_vline(xintercept = log10(c(ncount_min, ncount_max)), 
                             linetype = "dashed", color = "red") +
                  geom_hline(yintercept = log10(c(nfeature_min, nfeature_max)), 
                             linetype = "dashed", color = "red") +
                  scale_color_manual(values = c("TRUE" = "#06D6A0", "FALSE" = "#EF476F")) +
                  
                  labs(
                    title = "Cell Filtering Thresholds",
                    subtitle = paste0(
                      sum(qc_df$pass_qc), " cells pass QC (",
                      round(sum(qc_df$pass_qc) / nrow(qc_df) * 100, 1), "%). ",
                      "If thresholds are applied, ",
                      nrow(qc_df) - sum(qc_df$pass_qc),
                      " cells (",
                      round((1 - sum(qc_df$pass_qc) / nrow(qc_df)) * 100, 1),
                      "%) will be erased."
                    ),
                
                    x = "log10(UMI + 1)",
                    y = "log10(Genes + 1)",
                    color = "Pass QC") +  theme_classic()
    
    cat("Visual Credits go Dr. Li Guo of www.ngs101.com")
    
    threshold_image_path <- glue::glue("qc_images/{dataset_name}_threshold_checking.png")

    gg_patchwork(p8, threshold_image_path)
    
    return(list(image = p8))
}#end of function